# Extract realizations to be exported into JutulDarcy

* When using generate_full_properties(), set reverse_j = False  
* The finit files must contain ACTID to identify active cells, in addition to properties such as PORO and PERMX.
* This notebook has been validated by (1) processing a realization using this notebook, (2) import processed realiztion into Petrel, (3) compare it with the realization in Petrel and found statistics are exactly the same. Note when exporting a property out of Petrel, setting undefined cells to be 0 (instead of -999) would generate the exact same file as a processed realization by this notebook.

# Extract finit files from simulation cases

In [ ]:
from pathlib import Path
import shutil
import re
from tqdm import tqdm

################# Start of user inputs ########## 
data_folder_path = Path('../data/sim_cases') # path to the folder containing simulation case subfolders, each with a .FINIT file
results_folder_path = Path('../results/finit_files') # path to the folder where extracted .FINIT files will be copied. Will be created if it doesn't exist.
################## End of user inputs  ##########

# Create destination directory if it doesn't exist
results_folder_path.mkdir(parents=True, exist_ok=True)

finit_file_names = []

# Iterate through each simulation case folder
for case_path in tqdm(data_folder_path.iterdir(), desc='Extracting finit files'):
    if case_path.is_dir():
        for src_file in case_path.iterdir():
            if src_file.is_file() and src_file.suffix.upper() == '.FINIT':
                dst_file = results_folder_path / src_file.name
                shutil.copy2(src_file, dst_file)
                finit_file_names.append(src_file.name)

# Function to extract the trailing number
def extract_number(filename):
    match = re.search(r'(\d+)(?=\.FINIT$)', filename.upper())
    return int(match.group(1)) if match else float('inf')

# Sort numerically by extracted number
finit_file_names = sorted(finit_file_names, key=extract_number)

# Print 
print(f"Total FINIT files copied: {len(finit_file_names)}")
print("finite_file_names = [")
for fname in finit_file_names:
    print(f"    '{fname}',")
print("]")


Extracting finit files: 4it [00:00,  4.68it/s]

Total FINIT files copied: 3
finite_file_names = [
    'JD_BASECASE_10.FINIT',
    'JD_BASECASE_25.FINIT',
    'JD_BASECASE_72.FINIT',
]


# Extract finit file names from its folder

In [ ]:
from pathlib import Path
import re

################# Start of user inputs ########## 
folder_path = Path("../results/finit_files") # path to the folder containing .FINIT files extracted from simulation cases
################## End of user inputs  ##########


# Collect all file names (just the name, not full path)
finit_file_names = [f.name for f in folder_path.iterdir() if f.is_file()]

# Function to extract number inside filename
def extract_number(filename):
    match = re.search(r'(\d+)(?=\.FINIT$)', filename.upper())
    return int(match.group(1)) if match else float('inf')

# Sort numerically
finit_file_names = sorted(finit_file_names, key=extract_number)

# Print in the desired format
print("finite_file_names = [")
for fname in finit_file_names:
    print(f"    '{fname}',")
print("]")


finite_file_names = [
    'JD_BASECASE_10.FINIT',
    'JD_BASECASE_25.FINIT',
    'JD_BASECASE_72.FINIT',
]


# Extract properties

In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent
sys.path.append(str(repo_root / "src"))

from extract_properties_from_finit import extract_properties_from_finit
from generate_full_properties import generate_full_properties
from CMG_format_compress import CMG_format_compress

################# Start of user inputs ########## 
finit_folder_path = Path('../results/finit_files') # path to the folder containing .FINIT files extracted from simulation cases
save_folder_path = Path('../results/properties') # path to the folder where extracted properties will be saved. Will be created if it doesn't exist.
grid_shape = (139, 248, 23) # shape of the geomodel grid (i, j, k)
property_list = ['PORO', 'PERMX'] # list of property keywords to extract from
################## End of user inputs  ##########
save_folder_path.mkdir(exist_ok=True)

## extract finit file names from its folder 
# collect all file names (just the name, not full path)
finit_file_names = [f.name for f in finit_folder_path.iterdir() if f.is_file()]

# function to extract number inside filename
def extract_number(filename):
    match = re.search(r'(\d+)(?=\.FINIT$)', filename.upper())
    return int(match.group(1)) if match else float('inf')

# sort numerically
finit_file_names = sorted(finit_file_names, key=extract_number)

## extract properties
for finit_file_name in finit_file_names:
    finit_file_path = finit_folder_path / finit_file_name
    # some finit files contains PORO cells < active cells, so use try-except-continue to avoid interuption
    try:
        # STEP 1: Extract properties from FINIT file for active cells only
        extracted_property_dict = extract_properties_from_finit(
            finit_file_path = finit_file_path,
            keywords = property_list + ['ACTID'],
            is_save = False,
            save_dir = save_folder_path,
            save_name = finit_file_name.split('.')[0],
            show_summary = False
        )

        # STEP 2: Generate full properties for all cells (fill inactive cells with zeros)
        full_property_dict = generate_full_properties(
            property_dict = extracted_property_dict,
            property_list = property_list, 
            grid_shape = grid_shape,
            is_save = False,
            save_dir = save_folder_path,
            save_name = finit_file_name.split('.')[0],
            show_summary = False,
            reverse_j = False
            )


        # STEP 3: Compress full properties to CMG format (repeated values as N*value)
        for key in property_list:
            CMG_format_compress(
                array = full_property_dict[key], 
                keyword = key, 
                max_line_length = 80,
                show_summary = False,
                save_dir = save_folder_path,
                save_name = finit_file_name.split('.')[0]
            )
    except ValueError as e:
        print(f"Warning: Skipping '{finit_file_name}'. Reason: {e}")
        continue


Saved compressed PORO data to: ../results/properties/JD_BASECASE_10_PORO.dat
Saved compressed PERMX data to: ../results/properties/JD_BASECASE_10_PERMX.dat
Saved compressed PORO data to: ../results/properties/JD_BASECASE_25_PORO.dat
Saved compressed PERMX data to: ../results/properties/JD_BASECASE_25_PERMX.dat
Saved compressed PORO data to: ../results/properties/JD_BASECASE_72_PORO.dat
Saved compressed PERMX data to: ../results/properties/JD_BASECASE_72_PERMX.dat


# Collect property file names to be used for LSH sampling

In [6]:
import os
import re
from pathlib import Path
import numpy as np

"""
Extract file names from the results/properties folder.
Returns a list of file names (excluding hidden files like .DS_Store),
sorted numerically by the number inside the file name.
"""

# Define the path to the properties folder
properties_folder_path = Path('../results/properties')
save_path = Path('../results/property_file_names')

# Check if the directory exists
if not os.path.exists(properties_folder_path):
    print(f"Error: Directory '{properties_folder_path}' does not exist.")

# Use pathlib for better cross-platform compatibility
properties_dir = Path(properties_folder_path)

# Extract number from filename helper
def extract_number(filename):
    match = re.search(r"(\d+)", filename)
    return int(match.group(1)) if match else float('inf')

# Get all files, excluding hidden files
property_file_names = [
    file_path.name for file_path in properties_dir.iterdir()
    if file_path.is_file() and not file_path.name.startswith('.')
]

# Sort numerically by the number inside the filename
property_file_names.sort(key=extract_number)

# Save the list in a csv file
np.savetxt(save_path/'property_file_names_seed.csv', np.array(property_file_names), delimiter=',', fmt='%s')

print("Saved property file names in order:")
for fname in property_file_names:
    print(fname)


Saved property file names in order:
JD_BASECASE_10_PORO.dat
JD_BASECASE_10_PERMX.dat
JD_BASECASE_25_PORO.dat
JD_BASECASE_25_PERMX.dat
JD_BASECASE_72_PERMX.dat
JD_BASECASE_72_PORO.dat


In [5]:
property_file_names

['JD_BASECASE_10_PORO.dat',
 'JD_BASECASE_10_PERMX.dat',
 'JD_BASECASE_25_PORO.dat',
 'JD_BASECASE_25_PERMX.dat',
 'JD_BASECASE_72_PERMX.dat',
 'JD_BASECASE_72_PORO.dat']